In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import datetime

In [ ]:
# ==========================================
# 1. Load, Merge & Filter
# ==========================================
print("Loading Data...")
train = pd.read_csv('train.csv', parse_dates=['Date'], low_memory=False)
store = pd.read_csv('store.csv')

# Merge and filter: Open stores with Sales > 0
df = pd.merge(train, store, on='Store', how='left')
df = df[(df['Open'] == 1) & (df['Sales'] > 0)]

# Sort by Store/Date (Critical for Lag features)
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

Loading Data...


In [ ]:
# ==========================================
# 2. Temporal Features
# ==========================================
print("Creating Temporal Features...")
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# Duration since competition opened
df['CompetitionOpenSince'] = pd.to_datetime(dict(year=df.CompetitionOpenSinceYear.fillna(1900),
                                                 month=df.CompetitionOpenSinceMonth.fillna(1), day=1))
df['CompetitionDaysOpen'] = (df['Date'] - df['CompetitionOpenSince']).dt.days.clip(lower=0)

Creating Temporal Features...


In [ ]:
# ==========================================
# 3. Lags & Rolling Stats
# ==========================================
print("Creating Lags & Rolling Stats...")
grouped = df.groupby('Store')['Sales']

# Lags (1, 2, 7 days) - Shifted to avoid leakage
df['Sales_Lag1'] = grouped.shift(1)
df['Sales_Lag2'] = grouped.shift(2)
df['Sales_Lag7'] = grouped.shift(7)

# Rolling (7-day mean/std) - Shifted first
df['Sales_RollMean7'] = grouped.shift(1).rolling(7).mean()
df['Sales_RollStd7']  = grouped.shift(1).rolling(7).std()

# Drop initial NaNs caused by lags
df.dropna(subset=['Sales_Lag1', 'Sales_Lag7', 'Sales_RollMean7'], inplace=True)

Creating Lags & Rolling Stats...


In [ ]:
# ==========================================
# 4. Fourier Terms (Seasonality)
# ==========================================
print("Adding Fourier Terms...")
# Weekly (period=7)
df['day_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)

# Yearly (period=365)
day_of_year = df['Date'].dt.dayofyear
df['year_sin'] = np.sin(2 * np.pi * day_of_year / 365)
df['year_cos'] = np.cos(2 * np.pi * day_of_year / 365)

Adding Fourier Terms...


In [ ]:
# ==========================================
# 5. Preprocessing
# ==========================================
print("Encoding & Imputing...")

# Fill NaNs (Fix: Use assignment instead of inplace=True to avoid FutureWarning)
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())

# Fill remaining NaNs with 0
df.fillna(0, inplace=True)

# Label Encode Categoricals
categorical_cols = ['StateHoliday', 'StoreType', 'Assortment']
for col in categorical_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Normalize Numerical Features [0, 1]
num_cols = ['CompetitionDistance', 'CompetitionDaysOpen', 'Sales_RollMean7', 'Sales_RollStd7']
df[num_cols] = MinMaxScaler().fit_transform(df[num_cols])

Encoding & Imputing...


In [ ]:
# ==========================================
# 6. Time-Series Split
# ==========================================
print("Splitting Data (Train/Val/Test)...")

# Chronological Split: Test (Last 6 weeks), Validation (Previous 6 weeks)
max_date = df['Date'].max()
test_cut = max_date - pd.Timedelta(days=42)  # 6 weeks
val_cut  = test_cut - pd.Timedelta(days=42)  # 6 weeks

train_df = df[df['Date'] < val_cut]
val_df   = df[(df['Date'] >= val_cut) & (df['Date'] < test_cut)]
test_df  = df[df['Date'] >= test_cut]

# Define Features & Target
target = 'Sales'
ignore_cols = ['Id', 'Date', 'Customers', 'Open', 'CompetitionOpenSince',
               'CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth']
features = [c for c in df.columns if c not in ignore_cols + [target]]

# Detailed Output for Verification
print("-" * 30)
print(f"Training Set:   {train_df.shape[0]} rows ({train_df.Date.min().date()} to {train_df.Date.max().date()})")
print(f"Validation Set: {val_df.shape[0]} rows ({val_df.Date.min().date()} to {val_df.Date.max().date()})")
print(f"Test Set:       {test_df.shape[0]} rows ({test_df.Date.min().date()} to {test_df.Date.max().date()})")
print("-" * 30)
print(f"Final Features ({len(features)}): \n{features}")

# Create Final X and y matrices
X_train, y_train = train_df[features], train_df[target]
X_val, y_val     = val_df[features], val_df[target]
X_test, y_test   = test_df[features], test_df[target]

print("\nPhase 2 Completed Successfully.")

Splitting Data (Train/Val/Test)...
------------------------------
Training Set:   280494 rows (2014-05-24 to 2015-04-23)
Validation Set: 36376 rows (2015-04-24 to 2015-06-04)
Test Set:       41415 rows (2015-06-05 to 2015-07-17)
------------------------------
Final Features (27): 
['Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'CompetitionDaysOpen', 'Sales_Lag1', 'Sales_Lag2', 'Sales_Lag7', 'Sales_RollMean7', 'Sales_RollStd7', 'day_sin', 'day_cos', 'year_sin', 'year_cos']

Phase 2 Completed Successfully.


In [ ]:
# ==========================================
# Phase 3 - Cell 1: Setup + Metrics + Linear Regression (From Scratch)
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# --------- Metrics ---------
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


# --------- Linear Regression (Closed-form) from scratch ---------
class LinearRegressionScratch:
    """
    Linear Regression using Normal Equation:
        w = (X^T X + lambda*I)^(-1) X^T y
    Supports optional L2 regularization (ridge) via l2.
    """
    def __init__(self, fit_intercept=True, l2=0.0):
        self.fit_intercept = fit_intercept
        self.l2 = float(l2)
        self.w = None  # weights

    def _add_intercept(self, X):
        if not self.fit_intercept:
            return X
        ones = np.ones((X.shape[0], 1), dtype=X.dtype)
        return np.hstack([ones, X])

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).reshape(-1, 1)
        Xb = self._add_intercept(X)

        # Regularization matrix (do NOT regularize intercept)
        I = np.eye(Xb.shape[1])
        if self.fit_intercept:
            I[0, 0] = 0.0

        A = Xb.T @ Xb + self.l2 * I
        b = Xb.T @ y

        # Use pseudo-inverse for numerical stability
        self.w = np.linalg.pinv(A) @ b
        return self

    def predict(self, X):
        if self.w is None:
            raise ValueError("Model is not fitted yet. Call fit() first.")
        X = np.asarray(X, dtype=np.float64)
        Xb = self._add_intercept(X)
        y_pred = Xb @ self.w
        return y_pred.reshape(-1)


print("Phase 3 initialized: RMSE + LinearRegressionScratch ready.")

Phase 3 initialized: RMSE + LinearRegressionScratch ready.


In [ ]:
# ==========================================
# Phase 3 - Cell 2: Train & Evaluate Baseline Linear Regression
# ==========================================

print("Preparing data for Linear Regression...")

# Remove any non-numeric columns from features
numeric_features = []
for col in features:
    if np.issubdtype(train_df[col].dtype, np.number):
        numeric_features.append(col)

print(f"Using {len(numeric_features)} numeric features.")

# Create final matrices
X_train = train_df[numeric_features].values
y_train = train_df[target].values

X_val = val_df[numeric_features].values
y_val = val_df[target].values

X_test = test_df[numeric_features].values
y_test = test_df[target].values


# Train model
print("Training Linear Regression (from scratch)...")
model = LinearRegressionScratch(fit_intercept=True, l2=0.0)
model.fit(X_train, y_train)

# Predictions
train_pred = model.predict(X_train)
val_pred   = model.predict(X_val)
test_pred  = model.predict(X_test)

# Evaluation
print("\nRMSE Results:")
print(f"Train RMSE: {rmse(y_train, train_pred):.4f}")
print(f"Val   RMSE: {rmse(y_val, val_pred):.4f}")
print(f"Test  RMSE: {rmse(y_test, test_pred):.4f}")

Preparing data for Linear Regression...
Using 26 numeric features.
Training Linear Regression (from scratch)...

RMSE Results:
Train RMSE: 1509.5418
Val   RMSE: 1574.1641
Test  RMSE: 1415.1475


In [ ]:
# ==========================================
# Phase 3 - Cell 3: XGBoost (2nd-order) from scratch + Tuning
# ==========================================

import numpy as np

# ---------- Helpers ----------
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


# ---------- Approx XGBoost Tree (Histogram-based) ----------
class _XGBNode:
    __slots__ = ("is_leaf", "value", "feature", "threshold", "left", "right")
    def __init__(self, is_leaf=False, value=0.0, feature=None, threshold=None, left=None, right=None):
        self.is_leaf = is_leaf
        self.value = float(value)
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right


class XGBTreeRegScratch:
    """
    One regression tree using XGBoost objective with 2nd-order (g,h).
    - Uses histogram binning to search splits efficiently.
    - Depth-wise growth.
    """
    def __init__(self, max_depth=4, min_child_weight=50.0, reg_lambda=1.0, gamma=0.0,
                 n_bins=64, colsample=1.0, random_state=42):
        self.max_depth = int(max_depth)
        self.min_child_weight = float(min_child_weight)
        self.reg_lambda = float(reg_lambda)
        self.gamma = float(gamma)
        self.n_bins = int(n_bins)
        self.colsample = float(colsample)
        self.random_state = int(random_state)
        self._rng = np.random.default_rng(self.random_state)

        self.root = None
        self.bin_edges_ = None   # list of edges per feature (for binning)
        self.n_features_ = None

    def _leaf_weight(self, G, H):
        return -G / (H + self.reg_lambda)

    def _gain(self, G, H):
        return (G * G) / (H + self.reg_lambda)

    def _best_split_hist(self, Xb, g, h, feat_idx):
        """
        Xb: binned features (int bins)
        g,h: gradients/hessians
        feat_idx: list/array of feature indices to evaluate
        """
        G_total = np.sum(g)
        H_total = np.sum(h)

        base_score = self._gain(G_total, H_total)

        best_gain = -np.inf
        best_f = None
        best_bin = None

        n_samples = Xb.shape[0]

        for f in feat_idx:
            xb = Xb[:, f]

            # histogram: sum g/h per bin
            # bins range [0, n_bins-1]
            G_bin = np.zeros(self.n_bins, dtype=np.float64)
            H_bin = np.zeros(self.n_bins, dtype=np.float64)
            # accumulate
            # (loop is OK; can optimize later)
            for i in range(n_samples):
                b = xb[i]
                G_bin[b] += g[i]
                H_bin[b] += h[i]

            # prefix sums to evaluate splits between bins
            G_left = 0.0
            H_left = 0.0
            # try split after bin k (left: <=k, right: >k)
            for k in range(self.n_bins - 1):
                G_left += G_bin[k]
                H_left += H_bin[k]

                G_right = G_total - G_left
                H_right = H_total - H_left

                # min_child_weight constraint (XGBoost: min sum h in child)
                if H_left < self.min_child_weight or H_right < self.min_child_weight:
                    continue

                gain = (self._gain(G_left, H_left) + self._gain(G_right, H_right) - base_score) - self.gamma
                if gain > best_gain:
                    best_gain = gain
                    best_f = f
                    best_bin = k

        return best_f, best_bin, best_gain

    def _build(self, X, Xb, g, h, depth):
        # if stop => leaf
        G = np.sum(g); H = np.sum(h)
        leaf_val = self._leaf_weight(G, H)

        if depth >= self.max_depth or X.shape[0] < 2:
            return _XGBNode(is_leaf=True, value=leaf_val)

        # choose features (colsample)
        n_features = X.shape[1]
        if self.colsample < 1.0:
            k = max(1, int(np.ceil(self.colsample * n_features)))
            feat_idx = self._rng.choice(n_features, size=k, replace=False)
        else:
            feat_idx = np.arange(n_features)

        f, split_bin, gain = self._best_split_hist(Xb, g, h, feat_idx)

        # no good split
        if f is None or gain <= 0:
            return _XGBNode(is_leaf=True, value=leaf_val)

        # convert split bin to real threshold (edge)
        # We split as: X <= threshold goes left.
        # For bin k, threshold = bin_edges[f][k] (upper edge of bin k)
        thr = float(self.bin_edges_[f][split_bin])

        left_mask = X[:, f] <= thr
        if left_mask.sum() == 0 or left_mask.sum() == X.shape[0]:
            return _XGBNode(is_leaf=True, value=leaf_val)

        left = self._build(X[left_mask], Xb[left_mask], g[left_mask], h[left_mask], depth + 1)
        right = self._build(X[~left_mask], Xb[~left_mask], g[~left_mask], h[~left_mask], depth + 1)

        return _XGBNode(is_leaf=False, feature=f, threshold=thr, left=left, right=right)

    def fit(self, X, g, h):
        X = np.asarray(X, dtype=np.float64)
        g = np.asarray(g, dtype=np.float64).reshape(-1)
        h = np.asarray(h, dtype=np.float64).reshape(-1)

        self.n_features_ = X.shape[1]

        # build bin edges per feature using quantiles for approx bins
        self.bin_edges_ = []
        Xb = np.empty_like(X, dtype=np.int32)

        qs = np.linspace(0.0, 1.0, self.n_bins + 1)[1:-1]  # exclude 0 and 1
        for f in range(self.n_features_):
            col = X[:, f]
            # if constant, set edges to constant
            if np.all(col == col[0]):
                edges = np.array([col[0]] * (self.n_bins - 1), dtype=np.float64)
            else:
                edges = np.quantile(col, qs).astype(np.float64)
                edges = np.unique(edges)
                # ensure enough edges; pad if too few
                if edges.size < self.n_bins - 1:
                    # pad with repeated last edge
                    pad = np.full((self.n_bins - 1 - edges.size,), edges[-1], dtype=np.float64)
                    edges = np.concatenate([edges, pad])
                elif edges.size > self.n_bins - 1:
                    edges = edges[: self.n_bins - 1]

            self.bin_edges_.append(edges)

            # binning: digitize into [0..n_bins-1]
            # bins defined by edges; right=True => edge inclusive on right
            Xb[:, f] = np.digitize(col, edges, right=True).astype(np.int32)

        self.root = self._build(X, Xb, g, h, depth=0)
        return self

    def _predict_one(self, node, x):
        while not node.is_leaf:
            if x[node.feature] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.value

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        out = np.empty(X.shape[0], dtype=np.float64)
        for i in range(X.shape[0]):
            out[i] = self._predict_one(self.root, X[i])
        return out


# ---------- XGBoost Regressor (2nd-order) ----------
class XGBoostRegScratch:
    """
    Approximate XGBoost for regression with squared error.
    Objective:
      L = 1/2 (y - pred)^2
      g = pred - y
      h = 1
    """
    def __init__(self, n_estimators=30, learning_rate=0.1,
                 max_depth=3, min_child_weight=50.0,
                 reg_lambda=1.0, gamma=0.0,
                 subsample=0.5, colsample=0.5,
                 n_bins=16, random_state=42):
        self.n_estimators = int(n_estimators)
        self.learning_rate = float(learning_rate)
        self.max_depth = int(max_depth)
        self.min_child_weight = float(min_child_weight)
        self.reg_lambda = float(reg_lambda)
        self.gamma = float(gamma)
        self.subsample = float(subsample)
        self.colsample = float(colsample)
        self.n_bins = int(n_bins)
        self.random_state = int(random_state)
        self._rng = np.random.default_rng(self.random_state)

        self.base_score_ = 0.0
        self.trees_ = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).reshape(-1)
        n = X.shape[0]

        self.base_score_ = float(np.mean(y))
        pred = np.full(n, self.base_score_, dtype=np.float64)
        self.trees_ = []

        for t in range(self.n_estimators):
            # gradients/hessians for squared error
            g = (pred - y)          # dL/dpred
            h = np.ones_like(g)     # d2L/dpred2

            # row subsampling
            if self.subsample < 1.0:
                m = max(1, int(np.ceil(self.subsample * n)))
                idx = self._rng.choice(n, size=m, replace=False)
                X_fit, g_fit, h_fit = X[idx], g[idx], h[idx]
            else:
                X_fit, g_fit, h_fit = X, g, h

            tree = XGBTreeRegScratch(
                max_depth=self.max_depth,
                min_child_weight=self.min_child_weight,
                reg_lambda=self.reg_lambda,
                gamma=self.gamma,
                n_bins=self.n_bins,
                colsample=self.colsample,
                random_state=self.random_state + 1000 + t
            )
            tree.fit(X_fit, g_fit, h_fit)

            update = tree.predict(X)
            pred += self.learning_rate * update
            self.trees_.append(tree)

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        pred = np.full(X.shape[0], self.base_score_, dtype=np.float64)
        for tree in self.trees_:
            pred += self.learning_rate * tree.predict(X)
        return pred


# ---------- Tuning (keep small to run in Colab) ----------
print("Training Approx-XGBoost (from scratch) + hyperparameter tuning...")

Xtr, ytr = X_train, y_train
Xva, yva = X_val, y_val
Xte, yte = X_test, y_test

param_grid = [
    # Model A:
    {"n_estimators": 40, "learning_rate": 0.10, "max_depth": 3,
     "min_child_weight": 80.0, "reg_lambda": 1.5, "gamma": 0.0,
     "subsample": 0.5, "colsample": 0.5, "n_bins": 16},

    # Model B:
    {"n_estimators": 60, "learning_rate": 0.08, "max_depth": 3,
     "min_child_weight": 80.0, "reg_lambda": 2.0, "gamma": 0.0,
     "subsample": 0.5, "colsample": 0.5, "n_bins": 16},
]

best = {"val_rmse": np.inf, "params": None, "model": None}

for i, p in enumerate(param_grid, start=1):
    print(f"\nModel {i}/{len(param_grid)} | params={p}")

    model = XGBoostRegScratch(**p, random_state=42)
    model.fit(Xtr, ytr)

    tr_pred = model.predict(Xtr)
    va_pred = model.predict(Xva)

    tr_rmse = rmse(ytr, tr_pred)
    va_rmse = rmse(yva, va_pred)

    print(f"  Train RMSE: {tr_rmse:.4f}")
    print(f"  Val   RMSE: {va_rmse:.4f}")

    if va_rmse < best["val_rmse"]:
        best.update({"val_rmse": va_rmse, "params": p, "model": model})

print("\n" + "-" * 40)
print("Best params:", best["params"])
print(f"Best Val RMSE: {best['val_rmse']:.4f}")

best_model = best["model"]
te_pred = best_model.predict(Xte)
print(f"Test RMSE (best model): {rmse(yte, te_pred):.4f}")
print("-" * 40)

Training Approx-XGBoost (from scratch) + hyperparameter tuning...

Model 1/2 | params={'n_estimators': 40, 'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 80.0, 'reg_lambda': 1.5, 'gamma': 0.0, 'subsample': 0.5, 'colsample': 0.5, 'n_bins': 16}
  Train RMSE: 1480.4229
  Val   RMSE: 1550.5169

Model 2/2 | params={'n_estimators': 60, 'learning_rate': 0.08, 'max_depth': 3, 'min_child_weight': 80.0, 'reg_lambda': 2.0, 'gamma': 0.0, 'subsample': 0.5, 'colsample': 0.5, 'n_bins': 16}
  Train RMSE: 1427.8435
  Val   RMSE: 1516.1318

----------------------------------------
Best params: {'n_estimators': 60, 'learning_rate': 0.08, 'max_depth': 3, 'min_child_weight': 80.0, 'reg_lambda': 2.0, 'gamma': 0.0, 'subsample': 0.5, 'colsample': 0.5, 'n_bins': 16}
Best Val RMSE: 1516.1318
Test RMSE (best model): 1370.7410
----------------------------------------


In [ ]:
# ==========================================
# Phase 3 - Next Cell: Store-aware Time CV (Walk-Forward) for Approx-XGBoost (from scratch)
# ==========================================

print("Running Store-aware Walk-Forward CV...")

# --- config ---
N_FOLDS = 2
VAL_DAYS = 42
GAP_DAYS = 7
MIN_TRAIN_DAYS = 60

DATE_COL = "Date"
STORE_COL = "Store"

best_params = best["params"] if "best" in globals() else {
    "n_estimators": 60, "learning_rate": 0.08, "max_depth": 3,
    "min_child_weight": 80.0, "reg_lambda": 2.0, "gamma": 0.0,
    "subsample": 0.5, "colsample": 0.5, "n_bins": 16
}

df_all = train_df.copy()
df_all = df_all.sort_values([STORE_COL, DATE_COL]).reset_index(drop=True)

global_max = df_all[DATE_COL].max()
fold_ends = [global_max - pd.Timedelta(days=VAL_DAYS * i) for i in range(N_FOLDS, 0, -1)]
fold_starts = [end - pd.Timedelta(days=VAL_DAYS) for end in fold_ends]

# --- store-aware fold split ---
def make_store_aware_folds(df, fold_starts, fold_ends, gap_days=0, min_train_days=180):
    folds = []
    stores = df[STORE_COL].unique()

    for fs, fe in zip(fold_starts, fold_ends):
        train_cut = fs - pd.Timedelta(days=gap_days)

        train_idx = []
        val_idx = []

        for s in stores:
            ds = df[df[STORE_COL] == s]
            # val window
            v = ds[(ds[DATE_COL] >= fs) & (ds[DATE_COL] < fe)]
            if len(v) == 0:
                continue

            # train history for that store
            t = ds[ds[DATE_COL] < train_cut]
            if len(t) == 0:
                continue

            # ensure enough time span for that store
            span_days = (t[DATE_COL].max() - t[DATE_COL].min()).days
            if span_days < min_train_days:
                continue

            train_idx.append(t.index.values)
            val_idx.append(v.index.values)

        if len(train_idx) == 0 or len(val_idx) == 0:
            folds.append((None, None, fs, fe))
        else:
            folds.append((np.concatenate(train_idx), np.concatenate(val_idx), fs, fe))

    return folds

folds = make_store_aware_folds(df_all, fold_starts, fold_ends, gap_days=GAP_DAYS, min_train_days=MIN_TRAIN_DAYS)

# --- run CV ---
fold_results = []
all_val_true = []
all_val_pred = []

for k, (tr_idx, va_idx, fs, fe) in enumerate(folds, start=1):
    print(f"\nFold {k}/{N_FOLDS} | Val window: {fs.date()} to { (fe - pd.Timedelta(days=1)).date() }")

    if tr_idx is None:
        print("  Skipped (not enough data across stores for this fold).")
        continue

    tr = df_all.loc[tr_idx]
    va = df_all.loc[va_idx]

    numeric_features = [c for c in features if np.issubdtype(tr[c].dtype, np.number)]
    Xtr = tr[numeric_features].values
    ytr = tr[target].values
    Xva = va[numeric_features].values
    yva = va[target].values

    model = XGBoostRegScratch(**best_params, random_state=42)
    model.fit(Xtr, ytr)

    pred_tr = model.predict(Xtr)
    pred_va = model.predict(Xva)

    tr_rmse = rmse(ytr, pred_tr)
    va_rmse = rmse(yva, pred_va)

    print(f"  Train RMSE: {tr_rmse:.4f}")
    print(f"  Val   RMSE: {va_rmse:.4f}  | n_val={len(yva)}")

    fold_results.append((k, fs.date(), (fe - pd.Timedelta(days=1)).date(), len(ytr), len(yva), tr_rmse, va_rmse))
    all_val_true.append(yva)
    all_val_pred.append(pred_va)

# --- aggregate CV RMSE across all folds (weighted by samples) ---
if len(all_val_true) > 0:
    y_all = np.concatenate(all_val_true)
    p_all = np.concatenate(all_val_pred)
    overall_rmse = rmse(y_all, p_all)
    print("\n" + "-" * 50)
    print(f"Overall Store-aware CV RMSE (all folds pooled): {overall_rmse:.4f}")
    print("-" * 50)
else:
    print("\nNo folds were executed. Try lowering MIN_TRAIN_DAYS or N_FOLDS.")

Running Store-aware Walk-Forward CV...

Fold 1/2 | Val window: 2014-12-18 to 2015-01-28
  Train RMSE: 1327.2425
  Val   RMSE: 2028.0795  | n_val=30890

Fold 2/2 | Val window: 2015-01-29 to 2015-03-11
  Train RMSE: 1432.7142
  Val   RMSE: 1334.1868  | n_val=40163

--------------------------------------------------
Overall Store-aware CV RMSE (all folds pooled): 1671.6277
--------------------------------------------------


In [ ]:
# ==========================================
# Phase 3 - SHAP Feature Importance (KernelExplainer)
# Works with our scratch model via predict function
# ==========================================

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

numeric_features = [c for c in features if np.issubdtype(train_df[c].dtype, np.number)]

np.random.seed(42)
bg_idx = np.random.choice(len(X_train), size=min(200, len(X_train)), replace=False)
X_bg = X_train[bg_idx]

explain_idx = np.random.choice(len(X_val), size=min(300, len(X_val)), replace=False)
X_explain = X_val[explain_idx]

def predict_fn(X):
    return best_model.predict(np.asarray(X, dtype=np.float64))

print("Building SHAP KernelExplainer (may take some time)...")
explainer = shap.KernelExplainer(predict_fn, X_bg)

print("Computing SHAP values...")
shap_values = explainer.shap_values(X_explain, nsamples=200)

# 5) Summary Plot
X_explain_df = pd.DataFrame(X_explain, columns=numeric_features)

shap.summary_plot(shap_values, X_explain_df, show=True)

In [ ]:
# ==========================================
# Phase 3 - Error Analysis (Test Set)
# ==========================================

print("Running Error Analysis on Test Set...")

import numpy as np
import pandas as pd

final_model = best_model

y_pred_test = final_model.predict(X_test)

analysis_df = test_df.copy()
analysis_df["Pred"] = y_pred_test
analysis_df["Error"] = analysis_df["Pred"] - analysis_df["Sales"]
analysis_df["AbsError"] = np.abs(analysis_df["Error"])
analysis_df["APE"] = analysis_df["AbsError"] / (analysis_df["Sales"] + 1e-6)

print("\nOverall Test RMSE:", rmse(analysis_df["Sales"], analysis_df["Pred"]))
print("Overall MAE:", analysis_df["AbsError"].mean())
print("Overall MAPE:", analysis_df["APE"].mean())

store_errors = (
    analysis_df.groupby("Store")["AbsError"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

print("\nTop 10 Stores with Highest MAE:")
print(store_errors)

promo_error = analysis_df.groupby("Promo")["AbsError"].mean()
print("\nAverage Absolute Error by Promo:")
print(promo_error)

school_error = analysis_df.groupby("SchoolHoliday")["AbsError"].mean()
print("\nAverage Absolute Error by SchoolHoliday:")
print(school_error)

dow_error = analysis_df.groupby("DayOfWeek")["AbsError"].mean()
print("\nAverage Absolute Error by DayOfWeek:")
print(dow_error)

holiday_error = analysis_df.groupby("StateHoliday")["AbsError"].mean()
print("\nAverage Absolute Error by StateHoliday:")
print(holiday_error)

worst_cases = analysis_df.sort_values("AbsError", ascending=False).head(20)
print("\nTop 20 Worst Individual Predictions:")
print(worst_cases[["Store", "Date", "Sales", "Pred", "AbsError", "Promo", "StateHoliday"]])

Running Error Analysis on Test Set...

Overall Test RMSE: 1370.7409853049353
Overall MAE: 906.4361422853486
Overall MAPE: 0.13954187727460016

Top 10 Stores with Highest MAE:
Store
1114    6646.727249
817     6162.793263
842     4919.229710
262     4766.087851
909     4488.323286
251     3855.461331
876     3456.044265
963     3020.563887
335     2941.812360
788     2735.910481
Name: AbsError, dtype: float64

Average Absolute Error by Promo:
Promo
0.0     824.414461
1.0    1015.028227
Name: AbsError, dtype: float64

Average Absolute Error by SchoolHoliday:
SchoolHoliday
0.0    893.591365
1.0    988.348481
Name: AbsError, dtype: float64

Average Absolute Error by DayOfWeek:
DayOfWeek
0    1448.533785
1     783.421974
2     684.913861
3     609.957216
4     663.638904
5    1230.992399
6    2887.603026
Name: AbsError, dtype: float64

Average Absolute Error by StateHoliday:
StateHoliday
0    906.436142
Name: AbsError, dtype: float64

Top 20 Worst Individual Predictions:
        Store      